In [ ]:
import sys
import numpy as np
import pandas as pd

CHEMIN = r"valeursfoncieres-2025.txt"
SORTIE = r"dvf_2025_mutations_propres.csv"

VALEUR_MIN = 1_000        # écarte les ventes symboliques / micro-ventes.
PRIX_M2_PLANCHER = 150     # prix/m² minimum (erreurs de saisie). Aucun plafond sur le prix/m².
PRIX_M2_PRO_PLANCHER = 50  # idem pour les locaux pro
SURFACE_MIN = 9            # surface habitable minimale.
Z_MAX, MIN_GROUPE_Z = 4.0, 20   # outlier relatif, CÔTÉ BAS UNIQUEMENT : z < -4, si le groupe a >= 20 ventes

# ───────────── 1. Lecture & renommage ─────────────
def lire(chemin):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(chemin, sep="|", dtype=str, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue

raw = lire(CHEMIN)
raw.columns = raw.columns.str.strip()

ren = {
    "No disposition": "no_disposition", "Date mutation": "date_mutation",
    "Nature mutation": "nature_mutation", "Valeur fonciere": "valeur_fonciere",
    "No voie": "no_voie", "B/T/Q": "btq", "Type de voie": "type_voie",
    "Code voie": "code_voie", "Voie": "voie", "Code postal": "code_postal",
    "Commune": "commune", "Code departement": "code_dept",
    "Code commune": "code_commune", "Prefixe de section": "prefixe_section",
    "Section": "section", "No plan": "no_plan", "No Volume": "no_volume",
    "Nombre de lots": "nb_lots", "Code type local": "code_type_local",
    "Type local": "type_local", "Surface reelle bati": "surface_bati",
    "Nombre pieces principales": "nb_pieces", "Nature culture": "nature_culture",
    "Nature culture speciale": "nature_culture_speciale",
    "Surface terrain": "surface_terrain",
}
for i, p in enumerate(["1er", "2eme", "3eme", "4eme", "5eme"], 1):
    ren[f"{p} lot"] = f"lot{i}"
    ren[f"Surface Carrez du {p} lot"] = f"carrez{i}"

# on ne garde que les colonnes utiles
df = raw.rename(columns=ren)[list(ren.values())].copy()
del raw
n_brut = len(df)

# ───────────── 2. Types & codes ─────────────
def num(s):
    return pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")

for c in ["valeur_fonciere", "surface_bati", "nb_pieces", "surface_terrain", "nb_lots"] \
         + [f"carrez{i}" for i in range(1, 6)]:
    df[c] = num(df[c])

dm = pd.to_datetime(df["date_mutation"], format="%d/%m/%Y", errors="coerce")
dm = dm.fillna(pd.to_datetime(df["date_mutation"], format="%Y-%m-%d", errors="coerce"))
df["date_mutation"] = dm
 
df["code_dept"] = df["code_dept"].str.zfill(2)
dom = df["code_dept"].str.len() == 3                       # DOM : commune sur 2 chiffres
df["code_insee"] = np.where(dom, df.code_dept + df.code_commune.str.zfill(2), df.code_dept + df.code_commune.str.zfill(3))
df["code_postal"] = df["code_postal"].str.zfill(5)
df["id_parcelle"] = (df.code_insee + df.prefixe_section.fillna("000").str.zfill(3) + df.section.str.zfill(2) + df.no_plan.str.zfill(4))

# ───────────── 2bis. Identifiant de mutation ─────────────
# Les lignes d'une même mutation sont contiguës dans le fichier source : on numérote les "runs" successifs, sans jamais trier df.
# La clé sans run ne serait PAS unique à l'échelle nationale (mêmes date / disposition / prix ronds dans des départements différents).
d = df.date_mutation.dt.strftime("%Y%m%d").fillna("NA")
cle = (d + "|" + df.no_disposition.fillna("") + "|" + df.valeur_fonciere.fillna(-1).astype(str) + "|" + df.code_dept)
run = (cle != cle.shift()).cumsum()
df["id_mutation"] = d + "-" + run.astype(str).str.zfill(8)
del d, cle, run

# ───────────── 3. Filtres (l'id est déjà attribué, filtrer ne le casse pas) ─────────────
df = df[df.nature_mutation.str.startswith("Vente", na=False)].copy()   # Vente, VEFA, terrain à bâtir
df = df[df.valeur_fonciere >= VALEUR_MIN].copy()                       # NaN et ventes < 15 000 € exclus ; pas de plafond
df = df[df.date_mutation.notna()].copy()
 
nm = df.nature_mutation.str.lower()
df["vefa"] = nm.str.contains("futur")
df["type_vente"] = np.select([nm.str.contains("futur"), nm.str.contains("terrain")], ["VEFA", "Terrain à bâtir"], default="Vente classique")
 
# type de local : on se fie au code (1 maison, 2 appartement, 3 dépendance, 4 local pro)
code = df.code_type_local.str.strip().str[:1].map(
    {"1": "maison", "2": "appartement", "3": "dependance", "4": "local_pro"})
t = df.type_local.fillna("").str.lower()
txt = pd.Series(np.select(
    [t.str.startswith("maison"), t.str.startswith("appartement"), t.str.startswith("dépendance") | t.str.startswith("dependance"), t.str.startswith("local")],
    ["maison", "appartement", "dependance", "local_pro"], default=""), index=df.index).replace("", np.nan)
df["local"] = code.fillna(txt)
 
NC = {"AB": "constructible",
      "S": "attenant", "J": "attenant", "AG": "attenant",
      "P": "agricole", "T": "agricole", "VI": "agricole", "VE": "agricole",
      "VG": "agricole", "PA": "agricole", "PC": "agricole", "L": "agricole",
      "BT": "foret", "BS": "foret", "BR": "foret", "BP": "foret", "BF": "foret", "LB": "foret"}

# ───────────── 4. Locaux : annuler le produit cartésien n locaux × p natures de culture ─────────────
loc = df[df.local.notna()].copy()
loc["ng_key"] = (loc.nature_culture.fillna("") + "|" + loc.nature_culture_speciale.fillna("") + "|" + loc.surface_terrain.fillna(-1).astype(str))   # fillna : astype(str) garde les NaN en pandas >= 3
loc["n_ng"] = loc.groupby(["id_mutation", "id_parcelle"]).ng_key.transform("nunique")
loc["lots_sig"] = (loc.lot1.fillna("") + "-" + loc.lot2.fillna("") + "-" + loc.lot3.fillna("") + "-" + loc.lot4.fillna("") + "-" + loc.lot5.fillna(""))
loc["nb_pieces"] = loc.nb_pieces.fillna(-1)
loc["surface_bati"] = loc.surface_bati.fillna(0)
 
KEYS = ["id_mutation", "id_parcelle", "local", "surface_bati", "nb_pieces", "lots_sig"]
loc_u = (loc.groupby(KEYS, as_index=False)
            .agg(n_rows=("ng_key", "size"), n_ng=("n_ng", "first"),
                 no_voie=("no_voie", "first"), btq=("btq", "first"),
                 type_voie=("type_voie", "first"), voie=("voie", "first"),
                 code_postal=("code_postal", "first"), commune=("commune", "first"),
                 code_insee=("code_insee", "first"), code_dept=("code_dept", "first")))
loc_u["n_ng"] = loc_u.n_ng.clip(lower=1)
loc_u["n_reel"] = np.maximum(1, np.rint(loc_u.n_rows / loc_u.n_ng)).astype(int)
loc_u["np_incoherent"] = (loc_u.n_rows % loc_u.n_ng) != 0        # règle n x p non respectée
loc_u["surf_tot"] = loc_u.surface_bati * loc_u.n_reel
 
COLS = ["maison", "appartement", "local_pro", "dependance"]
def pivot(val):
    return (loc_u.pivot_table(index="id_mutation", columns="local", values=val, aggfunc="sum", fill_value=0).reindex(columns=COLS, fill_value=0))
 
nb = pivot("n_reel").add_prefix("nb_")
sf = pivot("surf_tot").drop(columns="dependance").add_prefix("surf_")   # surface dépendance = 0 par définition
pieces = (loc_u.assign(p=np.where(loc_u.local.isin(["maison", "appartement"]), loc_u.nb_pieces.clip(lower=0) * loc_u.n_reel, 0)).groupby("id_mutation").p.sum().rename("nb_pieces"))
 
# adresse / commune = local principal (maison > appartement > local pro > dépendance)
prio = {"maison": 0, "appartement": 1, "local_pro": 2, "dependance": 3}
principal = (loc_u.assign(r=loc_u.local.map(prio)).sort_values(["id_mutation", "r", "surface_bati"], ascending=[True, True, False]).drop_duplicates("id_mutation").set_index("id_mutation"))
GEO = ["no_voie", "btq", "type_voie", "voie", "code_postal", "commune", "code_insee", "code_dept"]
geo = (df[["id_mutation", "surface_terrain"] + GEO].sort_values(["id_mutation", "surface_terrain"], ascending=[True, False]).drop_duplicates("id_mutation").set_index("id_mutation"))
geo.loc[principal.index, GEO] = principal[GEO]
geo["adresse"] = (geo.no_voie.fillna("") + geo.btq.fillna("") + " " + geo.type_voie.fillna("") + " " + geo.voie.fillna("")).str.replace(r"\s+", " ", regex=True).str.strip()
geo = geo[["adresse", "commune", "code_postal", "code_insee", "code_dept"]]
 
# ───────────── 5. Terrains : une seule fois par (parcelle, nature, surface) ─────────────
ter = (df.dropna(subset=["surface_terrain"]).drop_duplicates(["id_mutation", "id_parcelle", "nature_culture", "nature_culture_speciale", "surface_terrain"]).copy())
ter["cat"] = ter.nature_culture.map(NC).fillna("autre")
TCOLS = ["terrain_constructible", "terrain_attenant", "terrain_agricole", "terrain_foret", "terrain_autre"]
terrain = (ter.pivot_table(index="id_mutation", columns="cat", values="surface_terrain", aggfunc="sum", fill_value=0).add_prefix("terrain_").reindex(columns=TCOLS, fill_value=0))
terrain["terrain_total"] = terrain.sum(axis=1)
 
# ───────────── 6. Lots de copropriété & Carrez ─────────────
rows = []
for i in range(1, 6):
    x = df.loc[df[f"lot{i}"].notna(), ["id_mutation", "id_parcelle", "local", f"lot{i}", f"carrez{i}"]]
    x.columns = ["id_mutation", "id_parcelle", "local", "lot", "carrez"]
    rows.append(x)
lots = pd.concat(rows).sort_values("carrez")          # NaN en dernier : on garde la valeur renseignée
nb_lots = (lots.drop_duplicates(["id_mutation", "id_parcelle", "lot"]).groupby("id_mutation").size().rename("nb_lots_copro"))   # minimum : lots > 5 non restitués
carrez = (lots[lots.local != "dependance"].drop_duplicates(["id_mutation", "id_parcelle", "lot"]).groupby("id_mutation").carrez.sum(min_count=1).rename("surface_carrez"))  # filtre AVANT dédoublonnage : le lot est répété sur chaque ligne de local
 
# ───────────── 7. Table finale : 1 ligne = 1 mutation ─────────────
mut = (df.groupby("id_mutation")
         .agg(date_mutation=("date_mutation", "first"), nature_mutation=("nature_mutation", "first"),
              type_vente=("type_vente", "first"), valeur_fonciere=("valeur_fonciere", "first"),
              vefa=("vefa", "first"), nb_lignes_brutes=("id_parcelle", "size"),
              nb_parcelles=("id_parcelle", "nunique"), nb_communes=("code_insee", "nunique"))
         .join([geo, nb, sf, pieces, terrain, nb_lots, carrez]))
zero = list(nb.columns) + list(sf.columns) + TCOLS + ["terrain_total", "nb_pieces", "nb_lots_copro"]
mut[zero] = mut[zero].fillna(0)
mut[list(nb.columns) + ["nb_pieces", "nb_lots_copro"]] = \
    mut[list(nb.columns) + ["nb_pieces", "nb_lots_copro"]].astype(int)
mut["annee_mois"] = mut.date_mutation.dt.to_period("M").astype(str)
 
sh = mut.surf_maison + mut.surf_appartement
mut["surface_habitable"] = sh.where(sh > 0, mut.surface_carrez)   # à défaut : Carrez (VEFA)
mut.loc[mut.surface_habitable <= 0, "surface_habitable"] = np.nan
mut["a_dependance"] = mut.nb_dependance > 0
mut["terrain_renseigne"] = mut.terrain_total > 0   # False pour une maison = terrain INCONNU, pas 0 m²
 
# type de bien
nb_log = mut.nb_maison + mut.nb_appartement
# cabane sur un terrain : pas un parking mut.nb_dependance > 0, mut.vefa & (mut.surface_carrez > 0), mut.vefa]

conds = [(mut.nb_local_pro > 0) & (nb_log > 0), mut.nb_local_pro > 0, nb_log >= 2, mut.nb_maison == 1, mut.nb_appartement == 1, (mut.nb_dependance > 0) & (mut.terrain_total >= 500), mut.nb_dependance > 0, mut.vefa & (mut.surface_carrez > 0), mut.vefa]
choix = ["Mixte habitation/pro", "Local pro", "Plusieurs logements", "Maison", "Appartement", "Terrain avec dépendance", "Dépendance seule (parking/cave)", "Appartement neuf (VEFA)", "Lots VEFA (surface inconnue)"]
mut["type_bien"] = pd.Series(np.select(conds, choix, default=""), index=mut.index)
 
lab_ter = {"terrain_constructible": "Terrain à bâtir", "terrain_attenant": "Terrain non bâti (sol/jardin)", "terrain_agricole": "Terrain agricole", "terrain_foret": "Forêt / bois", "terrain_autre": "Terrain autre"}
dom_ter = mut[TCOLS].idxmax(axis=1).map(lab_ter).where(mut.terrain_total > 0, "Indéterminé")
tab = (mut.type_vente == "Terrain à bâtir") & ((mut.terrain_constructible + mut.terrain_attenant) > 0)
dom_ter = dom_ter.where(~tab, "Terrain à bâtir")      # pas pour de la forêt / du pré déclarés "à bâtir"
mut["type_bien"] = mut.type_bien.where(mut.type_bien != "", dom_ter)
 
famille = {"Maison": "Résidentiel", "Appartement": "Résidentiel", "Plusieurs logements": "Résidentiel",
           "Mixte habitation/pro": "Mixte", "Local pro": "Commercial",
           "Appartement neuf (VEFA)": "Neuf", "Lots VEFA (surface inconnue)": "Neuf",
           "Dépendance seule (parking/cave)": "Annexe", "Indéterminé": "Indéterminé"}
mut["famille"] = mut.type_bien.map(famille).fillna("Terrain")
 
# prix au m²
res = mut.famille.isin(["Résidentiel", "Neuf"])
mut["prix_m2_habitable"] = (mut.valeur_fonciere / mut.surface_habitable).where(res)
mut["prix_m2_pro"] = (mut.valeur_fonciere / mut.surf_local_pro.replace(0, np.nan)).where(mut.famille == "Commercial")
mut["prix_m2_terrain"] = (mut.valeur_fonciere / mut.terrain_total.replace(0, np.nan)).where(mut.famille == "Terrain")
 
# ───────────── 8. Flags qualité (on marque, on ne supprime pas) ─────────────
# a) lots identiques en taille mais de numéros différents : appartement compté deux fois ?
dup = loc_u[loc_u.local.isin(["maison", "appartement"]) & (loc_u.lots_sig.str.strip("-") != "")]
dup = dup[dup.duplicated(["id_mutation", "id_parcelle", "local", "surface_bati", "nb_pieces"], keep=False)]
mut["flag_lots_doublon_suspect"] = mut.index.isin(dup.id_mutation)
 
# b) règle n x p de la doc DVF non respectée
mut["flag_np_incoherent"] = mut.index.isin(loc_u.loc[loc_u.np_incoherent, "id_mutation"])
 
# c) multi-communes : normal pour terrains / maisons, suspect pour un appartement ou un immeuble
mut["flag_multi_communes"] = mut.nb_communes > 1
mut["flag_multi_communes_suspect"] = mut.flag_multi_communes & mut.type_bien.isin(["Appartement", "Plusieurs logements", "Mixte habitation/pro"])
 
# d) prix/m² aberrant : z-score robuste sur le log, par département x type de bien.
#    PLANCHER SEULEMENT : on écarte les prix trop bas, on garde les prix très élevés (marché de luxe, etc.).
ok = mut.prix_m2_habitable.notna()
lp = np.log(mut.prix_m2_habitable.where(ok).clip(lower=1))
grp = [mut.code_dept, mut.type_bien]
med = lp.groupby(grp).transform("median")
mad = (lp - med).abs().groupby(grp).transform("median").clip(lower=0.10)
n_grp = lp.groupby(grp).transform("count")
z = (lp - med) / (1.4826 * mad)
mut["flag_prix_m2_aberrant"] = ok & ((mut.prix_m2_habitable < PRIX_M2_PLANCHER) | ((z < -Z_MAX) & (n_grp >= MIN_GROUPE_Z)))
 
# e) surfaces : minimum seulement (une surface manquante ou < SURFACE_MIN reste signalée), aucune surface maximale
mut["flag_surface_douteuse"] = (mut.famille == "Résidentiel") & ~(mut.surface_habitable >= SURFACE_MIN)
un_appt = (mut.nb_appartement == 1) & (mut.nb_maison == 0) & mut.surface_carrez.notna()
mut["flag_ecart_surface"] = un_appt & ((mut.surface_habitable / mut.surface_carrez - 1).abs() > 0.25)
 
# e bis) prix/m² des locaux pro : même logique, plancher seulement, par département
okp = mut.prix_m2_pro.notna()
lpp = np.log(mut.prix_m2_pro.where(okp).clip(lower=1))
gp = mut.code_dept
medp = lpp.groupby(gp).transform("median")
madp = (lpp - medp).abs().groupby(gp).transform("median").clip(lower=0.15)
zp = (lpp - medp) / (1.4826 * madp)
np_grp = lpp.groupby(gp).transform("count")
mut["flag_prix_pro_aberrant"] = okp & ((mut.prix_m2_pro < PRIX_M2_PRO_PLANCHER) | ((zp < -Z_MAX) & (np_grp >= MIN_GROUPE_Z)) | ((mut.terrain_agricole + mut.terrain_foret) > 0))   # ferme / exploitation
mut["prix_m2_pro_fiable"] = mut.prix_m2_pro.where(okp & ~mut.flag_prix_pro_aberrant & (mut.nb_communes == 1))
 
# f) information (non bloquant) : maison vendue avec du terrain agricole / forêt / constructible
mut["flag_bati_plus_terrain_hors_attenant"] = (mut.famille == "Résidentiel") & \
    (mut.terrain_agricole + mut.terrain_foret + mut.terrain_constructible > 0)
 
# mutation exploitable pour compter, mesurer un ticket, une liquidité
mut["retenue_stats"] = ~(mut.flag_prix_m2_aberrant | mut.flag_surface_douteuse | mut.flag_lots_doublon_suspect | mut.flag_ecart_surface | mut.flag_np_incoherent | mut.flag_multi_communes_suspect)
 
# mutation "simple" : un seul bien principal, pas de terrain agricole/forêt/constructible, 1 commune
mut["mutation_simple"] = (mut.type_bien.isin(["Maison", "Appartement", "Appartement neuf (VEFA)"]) & (mut.nb_communes == 1) & ((mut.terrain_agricole + mut.terrain_foret + mut.terrain_constructible) == 0))
 
# prix/m² à utiliser pour notre segmentation : uniquement les mutations simples et propres
mut["prix_m2_fiable"] = mut.prix_m2_habitable.where(mut.retenue_stats & mut.mutation_simple & ~mut.flag_bati_plus_terrain_hors_attenant)
 
# ───────────── 9. Export ─────────────
ordre = ["date_mutation", "annee_mois", "nature_mutation", "type_vente", "vefa",
         "adresse", "commune", "code_postal", "code_insee", "code_dept",
         "type_bien", "famille", "valeur_fonciere",
         "nb_maison", "nb_appartement", "nb_local_pro", "nb_dependance", "a_dependance",
         "surf_maison", "surf_appartement", "surf_local_pro", "surface_carrez", "surface_habitable",
         "nb_pieces", "nb_lots_copro",
         "terrain_constructible", "terrain_attenant", "terrain_agricole", "terrain_foret",
         "terrain_autre", "terrain_total", "terrain_renseigne",
         "prix_m2_habitable", "prix_m2_pro", "prix_m2_terrain", "prix_m2_fiable", "prix_m2_pro_fiable",
         "nb_lignes_brutes", "nb_parcelles", "nb_communes",
         "flag_lots_doublon_suspect", "flag_np_incoherent", "flag_multi_communes",
         "flag_multi_communes_suspect", "flag_prix_m2_aberrant", "flag_surface_douteuse",
         "flag_ecart_surface", "flag_prix_pro_aberrant", "flag_bati_plus_terrain_hors_attenant",
         "mutation_simple", "retenue_stats"]
assert mut.index.is_unique
mut = mut[ordre].reset_index()
mut.to_csv(SORTIE, index=False)
 
# ───────────── 10. Contrôles ─────────────
pref = np.where(mut.code_dept.str.len() == 3, mut.code_insee.str[:3], mut.code_insee.str[:2])
print(f"{n_brut:,} lignes brutes → {len(df):,} après filtres → {len(mut):,} mutations")
print(f"dept ≠ préfixe commune : {np.mean(pref != mut.code_dept):.2%}   (doit être ≈ 0)")
print(f"multi-communes         : {mut.flag_multi_communes.mean():.2%}   (doit rester faible)")
print(f"retenue_stats          : {mut.retenue_stats.mean():.1%}")
print(f"prix/m² fiable         : {mut.prix_m2_fiable.notna().sum():,} mutations")
print(f"valeur min / max       : {mut.valeur_fonciere.min():,.0f} € / {mut.valeur_fonciere.max():,.0f} €   (min doit être >= {VALEUR_MIN:,})")
print(f"prix/m² fiable max     : {mut.prix_m2_fiable.max():,.0f} €/m²   (à regarder : cas extrêmes conservés)")
print(mut.type_bien.value_counts())
print(mut.filter(like="flag_").sum())

3,714,829 lignes brutes → 3,226,807 après filtres → 1,128,295 mutations
dept ≠ préfixe commune : 0.00%   (doit être ≈ 0)
multi-communes         : 0.74%   (doit rester faible)
retenue_stats          : 97.9%
prix/m² fiable         : 712,400 mutations
valeur min / max       : 15,000 € / 695,000,000 €   (min doit être >= 15,000)
prix/m² fiable max     : 2,200,855 €/m²   (à regarder : cas extrêmes conservés)
type_bien
Maison                             453283
Appartement                        356172
Terrain agricole                    53489
Local pro                           47680
Plusieurs logements                 41666
Dépendance seule (parking/cave)     38180
Lots VEFA (surface inconnue)        36486
Terrain à bâtir                     34518
Terrain non bâti (sol/jardin)       26474
Mixte habitation/pro                21231
Appartement neuf (VEFA)              6714
Forêt / bois                         5533
Terrain avec dépendance              3535
Indéterminé                          